# 01 — Data Extraction

**Objective:** Load the raw FIFA World Cup Players dataset, inspect its structure, and confirm that the source is suitable for the project.

**Data Source:** [Kaggle — FIFA World Cup Dataset](https://www.kaggle.com/datasets/abecklas/fifa-world-cup)  
**File:** `data/raw/Primary/WorldCupPlayers.csv`  
**Coverage:** FIFA World Cup 1930 – 2014 (all editions)

## 1.1 — Setup & Imports

In [ ]:
from pathlib import Path

import pandas as pd

# Resolve project root regardless of where the notebook is launched from
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)
print(f"Project root: {PROJECT_ROOT}")

## 1.2 — Load Raw Dataset

In [ ]:
RAW_PATH = PROJECT_ROOT / "data/raw/Primary/WorldCupPlayers.csv"
print(f"Loading from: {RAW_PATH}")
print(f"File exists : {RAW_PATH.exists()}")

df = pd.read_csv(RAW_PATH)
print(f"\nLoaded successfully — {df.shape[0]:,} rows × {df.shape[1]} columns")

## 1.3 — First Look (Head & Tail)

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

## 1.4 — Shape, Columns & Data Types

In [ ]:
print(f"Rows   : {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumn names: {list(df.columns)}")
print("\n--- Data Types ---")
print(df.dtypes)

In [ ]:
df.info()

## 1.5 — Null / Missing Value Analysis

In [ ]:
null_summary = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "non_null_count": df.notnull().sum(),
    "dtype": df.dtypes
})
null_summary

**Observations:**
- `Position` is **89% null** — it only records `GK` (Goalkeeper), `C` (Captain), and `GKC` (GK + Captain). All other players have no position label.
- `Event` is **76% null** — only populated when a player had a notable match event (goal, card, substitution).
- All other columns are fully populated (0 nulls).

## 1.6 — Unique Value Counts

In [ ]:
unique_summary = pd.DataFrame({
    "unique_count": [df[col].nunique() for col in df.columns],
    "sample_values": [df[col].dropna().unique()[:5].tolist() for col in df.columns]
}, index=df.columns)
unique_summary

**Key numbers:**
- **836 matches** across **101 unique rounds**
- **82 countries** have participated
- **7,663 unique player names** recorded
- **335 unique coaches**
- `Line-up` has only 2 values: `S` (Starting XI) and `N` (Not starting / Bench)
- `Position` has only 3 values: `GK`, `C`, `GKC`

## 1.7 — Descriptive Statistics

In [ ]:
df.describe(include="all").T

## 1.8 — Sample Rows from Different Eras

In [ ]:
# Earliest matches (1930 era — low RoundIDs)
print("=== Earliest matches (lowest RoundIDs — 1930 era) ===")
df.nsmallest(10, "RoundID")[["RoundID", "MatchID", "Team Initials", "Player Name", "Position", "Shirt Number", "Event"]]

In [ ]:
# Latest matches (2014 era — high RoundIDs)
print("=== Latest matches (highest RoundIDs — 2014 era) ===")
df.nlargest(10, "RoundID")[["RoundID", "MatchID", "Team Initials", "Player Name", "Position", "Shirt Number", "Event"]]

**Observations:**
- In early tournaments (1930s), `Shirt Number` is `0` (jerseys were not numbered yet).
- In recent tournaments (2014), shirt numbers range 1–23 as expected.
- `Event` encoding is consistent across eras: `G` = Goal, `Y` = Yellow, `R` = Red, etc.

## 1.9 — Duplicate Check

In [ ]:
dup_count = df.duplicated().sum()
print(f"Exact duplicate rows: {dup_count}")
print(f"Percentage: {dup_count / len(df) * 100:.2f}%")

if dup_count > 0:
    print("\nSample duplicates:")
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10))

**Finding:** 736 exact duplicate rows exist (~1.95%). These will be removed during the Cleaning phase (Notebook 02).

## 1.10 — Event Column Preview

In [ ]:
# Preview unique event codes
print("Event column — sample non-null values:")
print(df["Event"].dropna().unique()[:25])
print(f"\nTotal unique event strings: {df['Event'].nunique()}")

**Event notation decoding:**

| Code | Meaning | Example |
|------|---------|---------|
| `G` | Goal | `G40'` = Goal at 40th minute |
| `Y` | Yellow card | `Y32'` |
| `R` | Red card | `R70'` |
| `I` | Substitution — came IN | `I62'` |
| `O` | Substitution — went OUT | `O62'` |
| `P` | Penalty goal | `P42'` |
| `W` | Own goal | `W51'` |
| `MP` | Missed penalty | `MP90'` |
| `RSY` | Red card (second yellow) | `RSY45'` |

Multiple events per player are space-separated, e.g., `G43' G87'` = 2 goals.

## 1.11 — Extraction Summary

| Item | Value |
|------|-------|
| Dataset | `WorldCupPlayers.csv` |
| Source | Kaggle — FIFA World Cup Dataset |
| Rows | 37,784 |
| Columns | 9 |
| Coverage | FIFA World Cup 1930 – 2014 |
| Teams | 82 countries |
| Players | 7,663 unique |
| Matches | 836 |
| Duplicates | 736 (to be removed) |
| Key nulls | Position (89%), Event (76%) |

✅ **Conclusion:** The dataset is suitable for the project. It provides comprehensive player-level match data across 90+ years of World Cup history. Proceed to Notebook 02 (Cleaning).